# Capitolo 4 — Quanto fidarsi di un numero: cross-validation a 5 fold sul dataset dei tumori (§ 4.9)

In [1]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

def fai_modello(ingressi, nascosti=256, dropout=0.0):
    return nn.Sequential(nn.Linear(ingressi, nascosti), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(nascosti, nascosti), nn.ReLU(), nn.Dropout(dropout), nn.Linear(nascosti, 1))

def addestra(modello, Xtr, ytr, Xva, yva, epoche=300, lr=1e-3, weight_decay=0.0, pazienza=None, pos_weight=None):
    perdita_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight) if pos_weight else None)
    opt = torch.optim.Adam(modello.parameters(), lr=lr, weight_decay=weight_decay)
    Xtr, ytr, Xva, yva = map(torch.from_numpy, (Xtr, ytr, Xva, yva))
    storia = {"train": [], "val": []}; migliore = {"perdita": float("inf"), "pesi": None, "epoca": 0}; attesa = 0
    for epoca in range(epoche):
        modello.train(); opt.zero_grad()
        perdita = perdita_fn(modello(Xtr).squeeze(1), ytr); perdita.backward(); opt.step()
        modello.eval()
        with torch.no_grad(): perdita_val = nn.BCEWithLogitsLoss()(modello(Xva).squeeze(1), yva).item()
        storia["train"].append(perdita.item()); storia["val"].append(perdita_val)
        if perdita_val < migliore["perdita"]:
            migliore = {"perdita": perdita_val, "epoca": epoca, "pesi": {k: v.clone() for k, v in modello.state_dict().items()}}; attesa = 0
        else:
            attesa += 1
            if pazienza is not None and attesa >= pazienza: break
    modello.load_state_dict(migliore["pesi"])
    return storia, migliore

def probabilita(modello, X):
    modello.eval()
    with torch.no_grad(): return torch.sigmoid(modello(torch.from_numpy(X)).squeeze(1)).numpy()

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score
d = load_breast_cancer(); X = d.data.astype(np.float32); y = (1 - d.target).astype(np.float32)

In [2]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuratezze, recall = [], []
for i_tr, i_te in skf.split(X, y):
    Xa, Xv, ya, yv = train_test_split(X[i_tr], y[i_tr], test_size=0.15, random_state=42, stratify=y[i_tr])
    scaler = StandardScaler().fit(Xa)                     # scaler e split di validazione rifatti in ogni fold
    fissa_seme(42); m = fai_modello(30, dropout=0.5)
    addestra(m, scaler.transform(Xa).astype(np.float32), ya, scaler.transform(Xv).astype(np.float32), yv, weight_decay=1e-3, pazienza=30)
    pred = (probabilita(m, scaler.transform(X[i_te]).astype(np.float32)) > 0.5).astype(int)
    accuratezze.append(accuracy_score(y[i_te], pred)); recall.append(recall_score(y[i_te], pred))
print("accuratezza per fold:", [f"{a:.1%}" for a in accuratezze], f"→ {np.mean(accuratezze):.1%} ± {np.std(accuratezze):.1%}")
print("recall per fold:     ", [f"{a:.1%}" for a in recall], f"→ {np.mean(recall):.1%} ± {np.std(recall):.1%}")

accuratezza per fold: ['96.5%', '97.4%', '96.5%', '99.1%', '99.1%'] → 97.7% ± 1.2%
recall per fold:      ['97.7%', '93.0%', '92.9%', '100.0%', '97.6%'] → 96.2% ± 2.8%
